# Explore BigQuery Tables

A lightweight, read-only notebook for inspecting the production odds table. Edit the configuration cell as needed.

Authentication uses Google Application Default Credentials. Run `gcloud auth application-default login` once if needed.

In [2]:
import pandas as pd
from google.cloud import bigquery

PROJECT_ID = "odds-tracker-402301"
DATASET_ID = "nfl_data"
TABLE_ID = "daily_odds_data"
FULL_TABLE_ID = f"{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}"

client = bigquery.Client(project=PROJECT_ID)
print(FULL_TABLE_ID)

odds-tracker-402301.nfl_data.daily_odds_data


## Available tables

In [3]:
tables = list(client.list_tables(f"{PROJECT_ID}.{DATASET_ID}"))
pd.DataFrame({"table_id": [table.table_id for table in tables]})

,table_id
0,daily_odds_data
1,game_metadata
2,game_results


## Table schema

In [4]:
table = client.get_table(FULL_TABLE_ID)
pd.DataFrame(
    {"name": field.name, "type": field.field_type, "mode": field.mode}
    for field in table.schema
)

,name,type,mode
0,value,STRING,NULLABLE
1,odd,FLOAT,NULLABLE
2,bet_subgroup,STRING,NULLABLE
3,subgroup_value,FLOAT,NULLABLE
4,bet_id,INTEGER,NULLABLE
5,bet_name,STRING,NULLABLE
6,bookmaker_id,INTEGER,NULLABLE
7,book,STRING,NULLABLE
8,game_id,INTEGER,NULLABLE
9,update_date,DATETIME,NULLABLE


## Bounded sample

This query reads at most 100 rows.

In [5]:
sample_query = f"""
SELECT *
FROM `{FULL_TABLE_ID}`
ORDER BY update_date DESC
LIMIT 100
"""
sample = client.query(sample_query).to_dataframe()
sample.head()


/home/matt-christy/Documents/TechnicalProjects/sports-betting-odds-tracker/odds/lib/python3.12/site-packages/google/cloud/bigquery/table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,value,odd,bet_subgroup,subgroup_value,bet_id,bet_name,bookmaker_id,book,game_id,update_date
0,Under 20.5,1.86,Under,20.5,30,Over/Under 2nd Half,4,Bet365,7785,2023-12-31 13:03:02
1,Over 22.5,2.00,Over,22.5,30,Over/Under 2nd Half,4,Bet365,7785,2023-12-31 13:03:02
2,Home,1.83,Home,NaN,1,Home/Away,4,Bet365,7785,2023-12-31 13:03:02
3,Over 20.5,1.86,Over,20.5,30,Over/Under 2nd Half,4,Bet365,7785,2023-12-31 13:03:02
4,Draw -3,9.50,Draw,-3.0,6,Handicap Result,4,Bet365,7785,2023-12-31 13:03:02


## Counts by sport

In [6]:
counts_query = f"""
SELECT
  book,
  bet_name,
  COUNT(*) AS row_count,
  COUNT(DISTINCT game_id) AS games,
  MIN(update_date) AS first_update,
  MAX(update_date) AS last_update
FROM `{FULL_TABLE_ID}`
GROUP BY book, bet_name
ORDER BY row_count DESC
LIMIT 100
"""
client.query(counts_query).to_dataframe()


/home/matt-christy/Documents/TechnicalProjects/sports-betting-odds-tracker/odds/lib/python3.12/site-packages/google/cloud/bigquery/table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,book,bet_name,row_count,games,first_update,last_update
0,Bet365,Handicap Result,3123,102,2023-11-07 21:56:54,2023-12-31 13:03:02
1,Bet365,Over/Under,2640,102,2023-11-06 14:26:08,2023-12-31 13:03:02
2,Bet365,Over/Under 1st Half,2498,102,2023-11-06 14:26:08,2023-12-31 13:03:02
3,Bet365,Over/Under 2nd Half,2458,102,2023-11-08 22:04:53,2023-12-31 13:03:02
4,Bet365,Home/Away,1320,102,2023-11-06 14:26:08,2023-12-31 13:03:02


## Recent odds updates


In [7]:
recent_query = f"""
SELECT update_date, game_id, book, bet_name, bet_subgroup, subgroup_value, value, odd
FROM `{FULL_TABLE_ID}`
ORDER BY update_date DESC
LIMIT 100
"""
client.query(recent_query).to_dataframe()


/home/matt-christy/Documents/TechnicalProjects/sports-betting-odds-tracker/odds/lib/python3.12/site-packages/google/cloud/bigquery/table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,update_date,game_id,book,bet_name,bet_subgroup,subgroup_value,value,odd
0,2023-12-31 13:03:02,7785,Bet365,Over/Under 2nd Half,Under,20.5,Under 20.5,1.86
1,2023-12-31 13:03:02,7785,Bet365,Over/Under 2nd Half,Over,22.5,Over 22.5,2.00
2,2023-12-31 13:03:02,7785,Bet365,Home/Away,Home,NaN,Home,1.83
3,2023-12-31 13:03:02,7785,Bet365,Over/Under 2nd Half,Over,20.5,Over 20.5,1.86
4,2023-12-31 13:03:02,7785,Bet365,Handicap Result,Draw,-3.0,Draw -3,9.50
...,...,...,...,...,...,...,...,...
95,2023-12-31 13:02:51,7784,Bet365,Over/Under 2nd Half,Under,20.5,Under 20.5,1.90
96,2023-12-31 13:02:51,7784,Bet365,Handicap Result,Draw,-7.0,Draw -7,12.00
97,2023-12-31 13:02:51,7784,Bet365,Over/Under 2nd Half,Under,20.5,Under 20.5,1.90
98,2023-12-31 13:02:51,7784,Bet365,Over/Under 2nd Half,Under,20.5,Under 20.5,1.90
